# Chinese Weibo Sentiment Analysis
## Application to COVID-19 Vaccine Attitudes

Machine Learning Mini-Project

Team Members:
- Mayukh Pravin (PES2UG24CS271)
- Medha Singh (PES2UG24CS274)

Course: UE24CS352A - Machine Learning

## 1. Problem Statement

The objective of this project is to develop machine-learning models
for binary sentiment classification of Chinese Weibo posts and
demonstrate their application to COVID-19/vaccine-related posts.

The project compares traditional machine-learning approaches with
a deep-learning approach and evaluates their performance using
accuracy, precision, recall and F1-score.

## 2. Methodology

The project follows the following pipeline:

Raw Chinese Weibo Text
        ↓
Text Cleaning
        ↓
Chinese Word Segmentation using Jieba
        ↓
Feature Representation
        ↓
 ┌──────────────┬──────────────┐
 │              │              │
TF-IDF       TF-IDF         Token Sequences
 │              │              │
Naive Bayes    SVM           Bi-LSTM
 │              │              │
 └──────────────┴──────────────┘
                ↓
       Performance Evaluation
                ↓
       COVID/Vaccine Application

**Environment setup**

!pip install jieba scikit-learn pandas numpy matplotlib seaborn tensorflow openpyxl


In [ ]:
from pathlib import Path
import pandas as pd

# Works when VS Code starts in either the project root or the Notebook folder.
possible_paths = [
    Path("weibo_senti_100k.csv"),
    Path("Notebook") / "weibo_senti_100k.csv",
    Path(__file__).parent / "weibo_senti_100k.csv" if "__file__" in globals() else Path("__file_not_available__"),
]
csv_path = next((p for p in possible_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find weibo_senti_100k.csv. Put it beside this notebook in the Notebook folder."
    )
df = pd.read_csv(csv_path)
print("Loaded dataset from:", csv_path.resolve())

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df.iloc[:, 0].value_counts())

In [ ]:
# Check how many posts are related to COVID-19 / vaccination

keywords = [
    "疫苗", "新冠", "冠状病毒", "COVID", "covid",
    "接种", "打针", "病毒", "疫情", "防疫",
    "辉瑞", "科兴", "国药", "疫苗接种"
]

pattern = "|".join(keywords)

vaccine_df = df[
    df["review"].astype(str).str.contains(pattern, case=False, na=False)
].copy()

print("Total posts:", len(df))
print("COVID/vaccine-related posts:", len(vaccine_df))

print("\nClass distribution:")
print(vaccine_df["label"].value_counts())

print("\nSample posts:")
display(vaccine_df[["label", "review"]].head(20))

In [ ]:
# Create a clean copy of the full sentiment dataset

data = df[["label", "review"]].copy()

# Remove any accidental empty reviews
data = data.dropna(subset=["review"])

# Make sure labels are integers
data["label"] = data["label"].astype(int)

print("Total samples:", len(data))
print("\nClass distribution:")
print(data["label"].value_counts())

print("\nClass percentages:")
print(data["label"].value_counts(normalize=True) * 100)

In [ ]:
from sklearn.model_selection import train_test_split

# Take a balanced 30,000-sample subset
data_small, _ = train_test_split(
    data,
    train_size=30000,
    stratify=data["label"],
    random_state=42
)

print("Working dataset:", data_small.shape)
print(data_small["label"].value_counts())

**Environment setup**

!pip install jieba


In [ ]:
import jieba
import re

def clean_text(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove @mentions
    text = re.sub(r"@\S+", "", text)

    # Remove unnecessary special characters
    text = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", " ", text)

    # Chinese word segmentation
    words = jieba.lcut(text)

    # Remove very short tokens
    words = [word for word in words if len(word.strip()) > 1]

    return " ".join(words)

data_small["clean_text"] = data_small["review"].apply(clean_text)

display(data_small[["review", "clean_text"]].head(10))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Separate features and labels
X = data_small["clean_text"]
y = data_small["label"]

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# TF-IDF feature extraction
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\nTF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Train Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Predictions
nb_pred = nb_model.predict(X_test_tfidf)

# Evaluation
nb_accuracy = accuracy_score(y_test, nb_pred)
nb_precision = precision_score(y_test, nb_pred)
nb_recall = recall_score(y_test, nb_pred)
nb_f1 = f1_score(y_test, nb_pred)

print("NAIVE BAYES RESULTS")
print("-------------------")
print("Accuracy :", round(nb_accuracy, 4))
print("Precision:", round(nb_precision, 4))
print("Recall   :", round(nb_recall, 4))
print("F1 Score :", round(nb_f1, 4))

In [ ]:
from sklearn.svm import LinearSVC

# Train SVM
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train_tfidf, y_train)

# Predictions
svm_pred = svm_model.predict(X_test_tfidf)

# Evaluation
svm_accuracy = accuracy_score(y_test, svm_pred)
svm_precision = precision_score(y_test, svm_pred)
svm_recall = recall_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred)

print("SVM RESULTS")
print("-----------")
print("Accuracy :", round(svm_accuracy, 4))
print("Precision:", round(svm_precision, 4))
print("Recall   :", round(svm_recall, 4))
print("F1 Score :", round(svm_f1, 4))

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)

In [ ]:
# Use the same 30,000-sample dataset
lstm_text = data_small["clean_text"].values
lstm_labels = data_small["label"].values

# Train/test split
X_lstm_train, X_lstm_test, y_lstm_train, y_lstm_test = train_test_split(
    lstm_text,
    lstm_labels,
    test_size=0.20,
    random_state=42,
    stratify=lstm_labels
)

print("LSTM training samples:", len(X_lstm_train))
print("LSTM testing samples:", len(X_lstm_test))

In [ ]:
# Create tokenizer
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_lstm_train)

# Convert text to integer sequences
X_train_seq = tokenizer.texts_to_sequences(X_lstm_train)
X_test_seq = tokenizer.texts_to_sequences(X_lstm_test)

# Pad sequences to the same length
max_length = 80

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

vocab_size = min(20000, len(tokenizer.word_index) + 1)

print("Vocabulary size:", vocab_size)
print("Training shape:", X_train_pad.shape)
print("Testing shape:", X_test_pad.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    SpatialDropout1D
)
from tensorflow.keras.optimizers import Adam

# Make sure labels are numeric
y_lstm_train = np.array(y_lstm_train).astype("float32")
y_lstm_test = np.array(y_lstm_test).astype("float32")

# Build improved LSTM model
lstm_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SpatialDropout1D(0.2),

    Bidirectional(
        LSTM(64, return_sequences=False)
    ),

    Dropout(0.4),

    Dense(32, activation="relu"),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

In [ ]:
history = lstm_model.fit(
    X_train_pad,
    y_lstm_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    verbose=1
)

In [ ]:
lstm_prob = lstm_model.predict(X_test_pad)

lstm_pred = (lstm_prob >= 0.5).astype(int).flatten()

lstm_accuracy = accuracy_score(y_lstm_test, lstm_pred)
lstm_precision = precision_score(y_lstm_test, lstm_pred)
lstm_recall = recall_score(y_lstm_test, lstm_pred)
lstm_f1 = f1_score(y_lstm_test, lstm_pred)

print("IMPROVED LSTM RESULTS")
print("---------------------")
print("Accuracy :", round(lstm_accuracy, 4))
print("Precision:", round(lstm_precision, 4))
print("Recall   :", round(lstm_recall, 4))
print("F1 Score :", round(lstm_f1, 4))

print("\nPrediction distribution:")
print(pd.Series(lstm_pred).value_counts())

## 3. Results

Three models were evaluated:

1. Naive Bayes — used as a baseline probabilistic classifier.
2. SVM — used as a traditional supervised machine-learning classifier.
3. Bi-LSTM — used to capture sequential information in the Chinese text.

The Bi-LSTM achieved the highest overall performance, while SVM
was the strongest traditional machine-learning model.

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Naive Bayes", "SVM", "Bi-LSTM"],
    "Accuracy": [nb_accuracy, svm_accuracy, lstm_accuracy],
    "Precision": [nb_precision, svm_precision, lstm_precision],
    "Recall": [nb_recall, svm_recall, lstm_recall],
    "F1 Score": [nb_f1, svm_f1, lstm_f1]
})

results = results.round(4)

display(results)

In [ ]:
import matplotlib.pyplot as plt

results_plot = results.set_index("Model")

results_plot[["Accuracy", "Precision", "Recall", "F1 Score"]].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Performance Comparison of Sentiment Classification Models")
plt.ylabel("Score")
plt.ylim(0.7, 0.9)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_lstm_test, lstm_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

disp.plot()
plt.title("Bi-LSTM Confusion Matrix")
plt.show()

print("Confusion Matrix:")
print(cm)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_lstm_test,
    lstm_pred,
    target_names=["Negative", "Positive"]
))

In [ ]:
# Clean the vaccine-related posts
vaccine_df["clean_text"] = vaccine_df["review"].apply(clean_text)

# Convert text to sequences using our trained tokenizer
vaccine_sequences = tokenizer.texts_to_sequences(
    vaccine_df["clean_text"]
)

vaccine_padded = pad_sequences(
    vaccine_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

# Predict using our best Bi-LSTM model
vaccine_prob = lstm_model.predict(vaccine_padded, verbose=0)

vaccine_df["predicted_sentiment"] = (
    vaccine_prob >= 0.5
).astype(int).flatten()

# Convert labels to readable names
vaccine_df["sentiment"] = vaccine_df["predicted_sentiment"].map({
    0: "Negative",
    1: "Positive"
})

print("COVID/Vaccine-related posts analyzed:", len(vaccine_df))

print("\nPredicted sentiment:")
print(vaccine_df["sentiment"].value_counts())

display(
    vaccine_df[["review", "sentiment"]].head(15)
)

In [ ]:
vaccine_counts = vaccine_df["sentiment"].value_counts()

plt.figure(figsize=(7, 5))

vaccine_counts.plot(kind="bar")

plt.title("Predicted Sentiment of COVID-19/Vaccine-Related Posts")
plt.xlabel("Sentiment")
plt.ylabel("Number of Posts")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
def predict_sentiment(text):
    # Clean text
    cleaned = clean_text(text)

    # Convert to sequence
    sequence = tokenizer.texts_to_sequences([cleaned])

    # Pad sequence
    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

    # Prediction
    probability = float(lstm_model.predict(padded, verbose=0)[0][0])

    if probability >= 0.5:
        sentiment = "POSITIVE"
    else:
        sentiment = "NEGATIVE"

    return sentiment, probability


# Test example
text = "这个疫苗非常安全，我支持接种"
sentiment, probability = predict_sentiment(text)

print("Input:", text)
print("Prediction:", sentiment)
print("Confidence:", round(probability * 100, 2), "%")